# 04 — Reproduce Paper Figures

Reproduce key figures from Neve-Oz, Sherman & Raveh 2024.

**Key result**: Ring of pTCR enrichment at the periphery of tight-contact zones,
matching ZAP-70 super-resolution data (Figure 4).

**Prerequisites**: Posterior samples from notebook 03.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().parent.parent.parent
STORE = ROOT / "projects" / "tcr_signaling" / "store"

## pTCR radial profile

The key prediction: pTCR is enriched at the contact boundary due to the
interplay between CD45 exclusion (kinetic segregation) and Lck* decay.

In [ ]:
# Compute pTCR profile as a function of distance from contact center
# Using the analytical models directly for visualization

contact_radius = 1.0  # um
cd45_bulk_density = 300.0  # molecules/um^2
lck_decay_length = 0.07  # um (70 nm)
lck_activation_rate = 0.5
tcr_density = 150.0
phos_rate = 0.1
dephos_rate = 0.2

# CD45 boundary density (from kinetic segregation)
contact_fraction = np.pi * contact_radius**2 / (2.0**2)
cd45_boundary = cd45_bulk_density / (1.0 - contact_fraction)

# Lck* as function of distance from contact edge (inward)
r = np.linspace(0, contact_radius, 200)
dist_from_edge = contact_radius - r
lck_star = lck_activation_rate * cd45_boundary * np.exp(-dist_from_edge / lck_decay_length)

# pTCR fraction
ptcr = (phos_rate * lck_star) / (phos_rate * lck_star + dephos_rate)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(r, lck_star)
axes[0].set_xlabel("Distance from center (um)")
axes[0].set_ylabel("Lck* concentration (a.u.)")
axes[0].set_title("Active Lck profile")
axes[0].axvline(contact_radius, color='red', linestyle='--', alpha=0.5, label='Contact edge')
axes[0].legend()

axes[1].plot(r, ptcr)
axes[1].set_xlabel("Distance from center (um)")
axes[1].set_ylabel("pTCR fraction")
axes[1].set_title("pTCR enrichment profile")
axes[1].axvline(contact_radius, color='red', linestyle='--', alpha=0.5, label='Contact edge')
axes[1].legend()

# 2D ring visualization
theta = np.linspace(0, 2*np.pi, 100)
R, Theta = np.meshgrid(r, theta)
X = R * np.cos(Theta)
Y = R * np.sin(Theta)
dist_from_edge_2d = contact_radius - R
lck_2d = lck_activation_rate * cd45_boundary * np.exp(-dist_from_edge_2d / lck_decay_length)
ptcr_2d = (phos_rate * lck_2d) / (phos_rate * lck_2d + dephos_rate)

im = axes[2].pcolormesh(X, Y, ptcr_2d, cmap='hot', shading='auto')
axes[2].set_xlabel("x (um)")
axes[2].set_ylabel("y (um)")
axes[2].set_title("pTCR spatial pattern (ring)")
axes[2].set_aspect('equal')
plt.colorbar(im, ax=axes[2], label='pTCR fraction')

plt.tight_layout()
plt.savefig(str(ROOT / 'projects/tcr_signaling/artifacts/ptcr_ring_profile.png'), dpi=150)
plt.show()
print("Key result: pTCR is enriched at the contact periphery (ring pattern)")

## Parameter sensitivity

How does the pTCR ring pattern change with Lck decay length?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for decay in [0.03, 0.05, 0.07, 0.10, 0.15]:
    lck = lck_activation_rate * cd45_boundary * np.exp(-dist_from_edge / decay)
    ptcr_profile = (phos_rate * lck) / (phos_rate * lck + dephos_rate)
    ax.plot(r, ptcr_profile, label=f"{decay*1000:.0f} nm")

ax.set_xlabel("Distance from center (um)")
ax.set_ylabel("pTCR fraction")
ax.set_title("pTCR profile sensitivity to Lck decay length")
ax.axvline(contact_radius, color='gray', linestyle='--', alpha=0.3)
ax.legend(title="Decay length")
plt.tight_layout()
plt.show()